# NSL-KDD : comparaison Cross-Entropy vs Focal Loss

### Connexion && Synchronisation Drive - Github

In [1]:
import os
import json
import urllib.request
from pathlib import Path

BENCHMARK_NAME = 'nsl-kdd'
os.environ['BENCHMARK_NAME'] = BENCHMARK_NAME
SETUP_URL = 'https://raw.githubusercontent.com/Appolinairee/-research-scratch-deep-learning-vision/main/notebooks/colab/setup_colab.ipynb'
SETUP_PATH = Path('/content/setup_colab.ipynb')
urllib.request.urlretrieve(SETUP_URL, SETUP_PATH)
setup_notebook = json.loads(SETUP_PATH.read_text())
for setup_cell in setup_notebook['cells']:
    if setup_cell['cell_type'] == 'code':
        exec(compile(''.join(setup_cell['source']), str(SETUP_PATH), 'exec'), globals())


Mounted at /content/drive
Projet synchronisé : /content/drive/MyDrive/learning-cifar10-from-scratch
Commit : f686727
Données : /content/drive/MyDrive/learning-cifar10-from-scratch/data/nsl-kdd
Modèles : /content/drive/MyDrive/learning-cifar10-from-scratch/models/nsl-kdd


In [ ]:
import subprocess
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data import NSLKDD_CLASSES, load_nsl_kdd
from src.experiments import (
    NSLKDDModelConfig,
    run_nsl_kdd_comparison,
)
from src.core.backend import get_backend

np.set_printoptions(precision=4, suppress=True)


In [ ]:
subprocess.run(
    ['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', 'main'],
    check=True,
)

local_commit = subprocess.check_output(
    ['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'],
    text=True,
).strip()
remote_commit = subprocess.check_output(
    ['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'origin/main'],
    text=True,
).strip()

print('Drive :', local_commit)
print('GitHub:', remote_commit)
print('Aligné :', local_commit == remote_commit)


## NSL-KDD - Network intrusion benchmark

Charger NSL-KDD et encoder ses 41 variables réseau pour le MLP.

**Pourquoi** : reproduire le benchmark DNN de l’article avec notre implémentation dense from scratch.

In [ ]:
import urllib.request

NSL_KDD_URLS = {
    'train': 'https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain%2B.txt',
    'test': 'https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest%2B.txt',
}
for split, url in NSL_KDD_URLS.items():
    destination = DATA_ROOT / f'KDD{split.title()}%2B.txt'
    if not destination.exists():
        urllib.request.urlretrieve(url, destination)
print('NSL-KDD files ready:', DATA_ROOT)

In [ ]:
data = load_nsl_kdd(
    train_path=DATA_ROOT / 'KDDTrain%2B.txt',
    test_path=DATA_ROOT / 'KDDTest%2B.txt',
    validation_fraction=0.2,
    seed=42,
)
print({
    'train': data.X_train.shape,
    'validation': data.X_validation.shape,
    'test': data.X_test.shape,
    'classes': NSLKDD_CLASSES,
})

In [ ]:
counts = np.bincount(data.y_train_idx, minlength=len(NSLKDD_CLASSES))
plt.figure(figsize=(8, 4))
plt.bar(NSLKDD_CLASSES, counts)
plt.title('NSL-KDD - distribution des classes')
plt.ylabel("Nombre d'exemples")
plt.show()

## Configuration de l’expérience

Les opérations de chargement, encodage, normalisation et séparation sont encapsulées dans `src.data.nsl_kdd`.

In [ ]:
config = NSLKDDModelConfig()
BACKEND = 'gpu'
backend = get_backend(BACKEND)

print({
    'backend': 'CuPy' if backend.is_gpu else 'NumPy',
    'architecture': [data.X_train.shape[1], *config.hidden_units, len(NSLKDD_CLASSES)],
    'losses': ['cross_entropy', 'focal_loss'],
    'adam_lr': config.learning_rate,
    'adam_decay': 0.004,
    'epochs': config.epochs,
    'batch_size': config.batch_size,
    'seed': 42,
})


## Entraînement comparatif

In [ ]:
comparison = run_nsl_kdd_comparison(
    data=data,
    config=config,
    backend=backend,
    seed=42,
)

for name, result in comparison.items():
    print({
        'loss': name,
        'elapsed_seconds': round(result.elapsed_seconds, 2),
        'best_epoch': result.history.best_epoch,
    })


## Courbes d’apprentissage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
colors = {'cross_entropy': 'tab:blue', 'focal_loss': 'tab:orange'}
labels = {'cross_entropy': 'Cross-entropy', 'focal_loss': 'Focal loss'}

for name, result in comparison.items():
    epochs = [row.epoch for row in result.history]
    axes[0].plot(epochs, [row.loss for row in result.history],
                 color=colors[name], label=f'{labels[name]} — train')
    axes[0].plot(epochs, [row.validation_loss for row in result.history],
                 color=colors[name], linestyle='--', label=f'{labels[name]} — validation')
    axes[1].plot(epochs, [row.accuracy for row in result.history],
                 color=colors[name], label=f'{labels[name]} — train')
    axes[1].plot(epochs, [row.validation_accuracy for row in result.history],
                 color=colors[name], linestyle='--', label=f'{labels[name]} — validation')

for ax, title, ylabel in zip(
    axes,
    ('Loss', 'Accuracy'),
    ('Loss', 'Accuracy'),
):
    ax.set_title(title)
    ax.set_xlabel('Époque')
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.25)
    ax.legend(frameon=False, ncol=2)
plt.tight_layout()
plt.show()


## Évaluation finale sur le test officiel

In [ ]:
global_rows = [
    {
        'Loss': name.replace('_', '-'),
        'Accuracy': result.report.accuracy,
        'Macro-precision': result.report.macro_precision,
        'Macro-recall': result.report.macro_recall,
        'Macro-F1': result.report.macro_f1,
    }
    for name, result in comparison.items()
]
global_results = pd.DataFrame(global_rows).set_index('Loss')
global_results.assign(**{column: (global_results[column] * 100).round(2) for column in global_results.columns}).rename(columns=lambda column: f'{column} (%)' if column != 'Loss' else column)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (name, result) in zip(axes, comparison.items()):
    matrix = result.report.confusion_matrix
    row_totals = matrix.sum(axis=1, keepdims=True)
    normalized = np.divide(matrix, row_totals, out=np.zeros_like(matrix, dtype=float), where=row_totals != 0)
    image = ax.imshow(normalized, cmap='Blues', vmin=0, vmax=1)
    for row in range(len(NSLKDD_CLASSES)):
        for column in range(len(NSLKDD_CLASSES)):
            ax.text(column, row, f'{normalized[row, column]:.2f}', ha='center', va='center', fontsize=8)
    ax.set_title(name.replace('_', ' ').title())
    ax.set_xlabel('Classe prédite')
    ax.set_ylabel('Classe réelle')
    ax.set_xticks(range(len(NSLKDD_CLASSES)), NSLKDD_CLASSES, rotation=45)
    ax.set_yticks(range(len(NSLKDD_CLASSES)), NSLKDD_CLASSES)
fig.suptitle('Matrices de confusion normalisées par classe réelle')
fig.colorbar(image, ax=axes, fraction=0.025, label='Proportion')
plt.tight_layout()
plt.show()


### Métriques par classe

In [ ]:
class_rows = [
    {
        'Loss': name.replace('_', '-'),
        'Class': class_name,
        'Precision': result.report.precision[class_index],
        'Recall': result.report.recall[class_index],
        'F1': result.report.f1[class_index],
    }
    for name, result in comparison.items()
    for class_index, class_name in enumerate(NSLKDD_CLASSES)
]
per_class_results = pd.DataFrame(class_rows).set_index(['Loss', 'Class'])
per_class_results.assign(**{column: (per_class_results[column] * 100).round(2) for column in per_class_results.columns}).rename(columns=lambda column: f'{column} (%)')


## Analyse post-entraînement

Tous les résultats ci-dessous proviennent des objets `comparison["cross_entropy"]` et `comparison["focal_loss"]` produits par la même fonction d’entraînement.


In [ ]:
preprocessing_audit = pd.DataFrame([
    ('Variables brutes', 41),
    ('Variables après encodage', data.X_train.shape[1]),
    ('Colonnes catégorielles', '1, 2, 3'),
    ('Validation', '20 % du train'),
    ('Seed du split', 42),
    ('Normalisation', 'moyenne et écart-type du train'),
], columns=['Élément', 'Valeur']).set_index('Élément')
preprocessing_audit


In [ ]:
protocol_audit = pd.DataFrame([
    ('Optimiseur', 'Adam'),
    ('Taux d’apprentissage', config.learning_rate),
    ('Decay Adam', 0.004),
    ('Taille de batch', config.batch_size),
    ('Époques demandées', config.epochs),
    ('Early stopping', 'patience = 25'),
    ('Scheduler / L2', 'désactivés'),
    ('Pertes comparées', 'Cross-entropy, Focal loss'),
], columns=['Paramètre', 'Valeur']).set_index('Paramètre')
protocol_audit


In [ ]:
split_names = ('Train', 'Validation', 'Test')
split_labels = (data.y_train_idx, data.y_validation_idx, data.y_test_idx)
split_counts = np.array([
    np.bincount(labels, minlength=len(NSLKDD_CLASSES)) for labels in split_labels
])
split_shares = split_counts / split_counts.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
x = np.arange(len(NSLKDD_CLASSES))
width = 0.25
for index, (split_name, counts) in enumerate(zip(split_names, split_counts)):
    axes[0].bar(x + (index - 1) * width, counts, width, label=split_name)
for index, (split_name, shares) in enumerate(zip(split_names, split_shares)):
    axes[1].bar(x + (index - 1) * width, shares * 100, width, label=split_name)
for ax, title, ylabel in zip(
    axes,
    ('Effectifs par classe', 'Proportions par classe'),
    ('Nombre d’exemples', 'Part du split (%)'),
):
    ax.set_title(title)
    ax.set_xticks(x, NSLKDD_CLASSES)
    ax.set_ylabel(ylabel)
    ax.grid(axis='y', alpha=0.25)
    ax.legend(frameon=False)
fig.suptitle('Distribution des classes selon le split', y=1.03)
plt.tight_layout()
plt.show()


In [ ]:
train_counts = split_counts[0]
test_counts = split_counts[2]
train_shares = split_shares[0]
test_shares = split_shares[2]
shift = test_shares - train_shares

fig, ax = plt.subplots(figsize=(10, 4.5))
colors = np.where(shift >= 0, 'tab:red', 'tab:blue')
bars = ax.bar(NSLKDD_CLASSES, shift * 100, color=colors)
ax.axhline(0, color='black', linewidth=0.8)
for bar, value in zip(bars, shift * 100):
    ax.annotate(
        f'{value:+.1f} pp',
        (bar.get_x() + bar.get_width() / 2, value),
        xytext=(0, 5 if value >= 0 else -5),
        textcoords='offset points',
        ha='center',
        va='bottom' if value >= 0 else 'top',
    )
ax.set_title('Décalage de distribution : test officiel − train')
ax.set_ylabel('Variation en points de pourcentage')
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()


## Courbes précision-rappel

In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)
y_test_indices = np.asarray(data.y_test_idx)
colors = {'cross_entropy': 'tab:blue', 'focal_loss': 'tab:orange'}

for class_index, (class_name, ax) in enumerate(zip(NSLKDD_CLASSES, axes.flat)):
    y_binary = (y_test_indices == class_index).astype(int)
    for name, result in comparison.items():
        precision, recall, _ = precision_recall_curve(y_binary, result.probabilities[:, class_index])
        average_precision = average_precision_score(y_binary, result.probabilities[:, class_index])
        ax.plot(recall, precision, linewidth=2, color=colors[name],
                label=f'{name.replace("_", "-")} — AP {average_precision:.3f}')
    ax.axhline(y_binary.mean(), color='gray', linestyle='--', linewidth=1)
    ax.set_title(class_name)
    ax.set_xlabel('Rappel')
    ax.set_ylabel('Précision')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.25)
    ax.legend(frameon=False, fontsize=8)
axes.flat[-1].set_visible(False)
fig.suptitle('Courbes précision-rappel one-vs-rest — test officiel', y=1.02)
plt.tight_layout()
plt.show()


## Représentations cachées

In [ ]:
from sklearn.decomposition import PCA

hidden_by_name = {
    name: result.hidden_activations
    for name, result in comparison.items()
}
combined_hidden = np.vstack(list(hidden_by_name.values()))
pca = PCA(n_components=2, random_state=42)
combined_2d = pca.fit_transform(combined_hidden)
split_index = len(hidden_by_name['cross_entropy'])
projection_by_name = {
    'cross_entropy': combined_2d[:split_index],
    'focal_loss': combined_2d[split_index:],
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharex=True, sharey=True)
for ax, (name, result) in zip(axes, comparison.items()):
    hidden_2d = projection_by_name[name]
    for class_index, class_name in enumerate(NSLKDD_CLASSES):
        mask = y_test_indices == class_index
        ax.scatter(hidden_2d[mask, 0], hidden_2d[mask, 1], s=7, alpha=0.35, label=class_name)
    ax.set_title(name.replace('_', ' ').title())
    ax.set_xlabel('Composante principale 1')
    ax.set_ylabel('Composante principale 2')
    ax.grid(alpha=0.2)
    ax.legend(markerscale=2, frameon=False, fontsize=8)
fig.suptitle(f'PCA commune des activations cachées 128D — variance expliquée : {pca.explained_variance_ratio_.sum():.1%}')
plt.tight_layout()
plt.show()


## Sparsité des représentations cachées

In [ ]:
width = 0.38
x = np.arange(len(NSLKDD_CLASSES))
fig, ax = plt.subplots(figsize=(10, 4.5))
colors = {'cross_entropy': 'tab:blue', 'focal_loss': 'tab:orange'}

for offset, (name, result) in zip((-width / 2, width / 2), comparison.items()):
    zero_fraction_by_class = []
    for class_index in range(len(NSLKDD_CLASSES)):
        mask = y_test_indices == class_index
        zero_fraction_by_class.append(np.mean(result.hidden_activations[mask] <= 0))
    values = np.asarray(zero_fraction_by_class) * 100
    bars = ax.bar(x + offset, values, width, label=name.replace('_', ' ').title(), color=colors[name])
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, value + 1, f'{value:.0f}',
                ha='center', va='bottom', fontsize=8)

ax.set_title('Sparsité moyenne de la couche cachée 128D')
ax.set_ylabel('Activations nulles (%)')
ax.set_xticks(x, NSLKDD_CLASSES)
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.25)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()
